In [4]:
import json
from pathlib import Path
from datetime import datetime
from funciones_auxiliares import * 

# --- 3 SUBREDDITS DANIELA ---
mis_subreddits = ["travel", "RandomThoughts", "books", "unpopularopinion", "jobs", "LeagueOfLegends"] 

# Parámetros solicitados en el enunciado 
HILOS_POR_SUBREDDIT = 70
COMENTARIOS_POR_HILO_SUBMISSION = 150
COMENTARIOS_POR_HILO_COMMENT = 30

todas_submissions = []

# 1. Extraer los 60 hilos (submissions)
# Filtramos hilos que tengan al menos 100 comentarios para asegurar el corpus
todas_submissions = extract_submissions(
	"datos/RS_2025.zst", 
	mis_subreddits, 
	n_submissions=HILOS_POR_SUBREDDIT, 
	min_comments=COMENTARIOS_POR_HILO_SUBMISSION
)

# 2. Extraer todos los comentarios de golpe
extract_comments_for_submissions(
	"datos/RC_2025.zst", 
	todas_submissions, 
	num_comments=COMENTARIOS_POR_HILO_COMMENT
)

# 3 y 4. Separar la "superlista" y guardar en JSONs individuales
print("Generando archivos JSON individuales...")

for sub in mis_subreddits:
	# Filtramos la lista global para quedarnos solo con los hilos de ESTE subreddit
	submissions_del_sub = [s for s in todas_submissions if s.get('subreddit', '').lower() == sub.lower()]
	
	if not submissions_del_sub:
		print(f"⚠️ No se encontraron hilos suficientes para r/{sub}")
		continue

	# Estructurar el resultado
	resultado_final = {
		"subreddit": sub,
		"extraction_date": datetime.now().isoformat(),
		"num_submissions": len(submissions_del_sub),
		"total_comments": sum(len(s['comments']) for s in submissions_del_sub),
		"submissions": submissions_del_sub
	}

	# Guardar en JSON individual
	nombre_archivo = f"ejemplo_subreddit_{sub}.json"
	with open(nombre_archivo, 'w', encoding='utf-8') as f:
		json.dump(resultado_final, f, ensure_ascii=False, indent=2)

	print(f"✅ Archivo '{nombre_archivo}' generado con éxito.")


In [2]:
import json
from datetime import datetime

# Lista de los archivos que generaste anteriormente
archivos_json = ["ejemplo_subreddit_travel.json", "ejemplo_subreddit_RandomThoughts.json", "ejemplo_subreddit_unpopularopinion.json",
				  "ejemplo_subreddit_jobs.json", "ejemplo_subreddit_books.json", "ejemplo_subreddit_LeagueOfLegends.json"]

for archivo in archivos_json:
	try:
		with open(archivo, 'r', encoding='utf-8') as f:
			data = json.load(f)
		
		print(f"\n{'='*50}")
		print(f"📅 ANÁLISIS TEMPORAL: r/{data['subreddit']}")
		print(f"{'='*50}")

		# Extraer fechas de creación de las submissions (convertidas de UTC a datetime)
		# created_utc viene en los datos originales del volcado [cite: 58]
		fechas = [datetime.fromtimestamp(s['created_utc']) for s in data['submissions']]
		
		if fechas:
			fechas_ordenadas = sorted(fechas)
			primera = fechas_ordenadas[0]
			ultima = fechas_ordenadas[-1]
			rango_dias = (ultima - primera).days

			print(f"🔹 Primera publicación: {primera.strftime('%Y-%m-%d %H:%M')}")
			print(f"🔹 Última publicación:  {ultima.strftime('%Y-%m-%d %H:%M')}")
			print(f"🔹 Amplitud temporal:    {rango_dias} días")

			# Contar cuántas hay por día para ver la densidad
			dias = [f.strftime('%Y-%m-%d') for f in fechas]
			conteo_dias = {dia: dias.count(dia) for dia in set(dias)}
			
			print("\n📊 Distribución por días (Primeros 5 días detectados):")
			for dia in sorted(conteo_dias.keys())[:5]:
				print(f"   - {dia}: {conteo_dias[dia]} hilos")
			
			if rango_dias < 1:
				print("\n⚠️ ALERTA: Todos los hilos son del mismo día. Considera saltar registros en la extracción.")
			else:
				print("\n✅ El corpus presenta variedad temporal.")
		else:
			print("❌ No se encontraron fechas en las submissions.")

	except FileNotFoundError:
		print(f"⚠️ No se encontró el archivo: {archivo}")


📅 ANÁLISIS TEMPORAL: r/travel
🔹 Primera publicación: 2025-01-01 19:49
🔹 Última publicación:  2025-02-09 06:50
🔹 Amplitud temporal:    38 días

📊 Distribución por días (Primeros 5 días detectados):
   - 2025-01-01: 1 hilos
   - 2025-01-02: 2 hilos
   - 2025-01-03: 3 hilos
   - 2025-01-05: 4 hilos
   - 2025-01-06: 2 hilos

✅ El corpus presenta variedad temporal.

📅 ANÁLISIS TEMPORAL: r/RandomThoughts
🔹 Primera publicación: 2025-01-01 02:04
🔹 Última publicación:  2025-01-13 23:51
🔹 Amplitud temporal:    12 días

📊 Distribución por días (Primeros 5 días detectados):
   - 2025-01-01: 2 hilos
   - 2025-01-02: 7 hilos
   - 2025-01-03: 6 hilos
   - 2025-01-04: 4 hilos
   - 2025-01-05: 10 hilos

✅ El corpus presenta variedad temporal.

📅 ANÁLISIS TEMPORAL: r/unpopularopinion
🔹 Primera publicación: 2025-01-01 05:20
🔹 Última publicación:  2025-01-06 21:17
🔹 Amplitud temporal:    5 días

📊 Distribución por días (Primeros 5 días detectados):
   - 2025-01-01: 11 hilos
   - 2025-01-02: 13 hilos
   -

In [3]:
import json
import re


def analizar_calidad(texto):
	# Patrones para detectar URLs y Emails
	tiene_url = bool(re.search(r'https?://\S+|www\.\S+', texto))
	tiene_email = bool(re.search(r'\S+@\S+\.\S+', texto))
	tiene_eliminado = bool(re.search(r'\[removed\]|\[deleted\]', texto))
	longitud = len(texto.split()) # Contamos palabras
	return longitud, tiene_url, tiene_email, tiene_eliminado

total_inservibles = 0
total_general = 0

for archivo in archivos_json:
	with open(archivo, 'r', encoding='utf-8') as f:
		data = json.load(f)
	
	total_comentarios = 0
	cortos = 0 # Menos de 5 palabras
	solo_links = 0
	con_email = 0
	eliminados = 0
	
	for submission in data['submissions']:
		for comment in submission.get('comments', []):
			total_comentarios += 1
			cuerpo = comment.get('body', '')
			
			n_palabras, has_url, has_email, has_removed = analizar_calidad(cuerpo)
			
			if n_palabras < 5:
				cortos += 1
			if has_url and n_palabras < 3: # Muy corto y con URL suele ser solo spam/link
				solo_links += 1
			if has_email:
				con_email += 1
			if has_removed:
				eliminados += 1

	total_inservibles += cortos + solo_links + con_email + eliminados 
	total_general += total_comentarios
	print(f"\n{'='*50}")
	print(f"🔍 CALIDAD DEL TEXTO: r/{data['subreddit']}")
	print(f"{'='*50}")
	print(f"✅ Total analizados: {total_comentarios}")
	print(f"⚠️ Comentarios muy cortos (< 5 palabras): {cortos} ({cortos/total_comentarios*100:.1f}%)")
	print(f"🔗 Comentarios que son casi solo URLs: {solo_links}")
	print(f"📧 Comentarios con emails: {con_email}")
	print(f"🚫 Comentarios con eliminados: {eliminados}")
	
	if cortos / total_comentarios > 0.2:
		print("💡 Sugerencia: El corpus tiene mucho 'ruido' (mensajes cortos). Deberías filtrar en el siguiente paso.")

print(f'\nComentarios totales: {total_general}. Comentarios inservibles: {total_inservibles}. Comentarios útiles antes de detectar bots o spam: {total_general-total_inservibles}')


🔍 CALIDAD DEL TEXTO: r/travel
✅ Total analizados: 2100
⚠️ Comentarios muy cortos (< 5 palabras): 243 (11.6%)
🔗 Comentarios que son casi solo URLs: 5
📧 Comentarios con emails: 0
🚫 Comentarios con eliminados: 21

🔍 CALIDAD DEL TEXTO: r/RandomThoughts
✅ Total analizados: 2100
⚠️ Comentarios muy cortos (< 5 palabras): 565 (26.9%)
🔗 Comentarios que son casi solo URLs: 1
📧 Comentarios con emails: 0
🚫 Comentarios con eliminados: 55
💡 Sugerencia: El corpus tiene mucho 'ruido' (mensajes cortos). Deberías filtrar en el siguiente paso.

🔍 CALIDAD DEL TEXTO: r/unpopularopinion
✅ Total analizados: 2100
⚠️ Comentarios muy cortos (< 5 palabras): 230 (11.0%)
🔗 Comentarios que son casi solo URLs: 2
📧 Comentarios con emails: 0
🚫 Comentarios con eliminados: 73

🔍 CALIDAD DEL TEXTO: r/jobs
✅ Total analizados: 2100
⚠️ Comentarios muy cortos (< 5 palabras): 260 (12.4%)
🔗 Comentarios que son casi solo URLs: 2
📧 Comentarios con emails: 0
🚫 Comentarios con eliminados: 52

🔍 CALIDAD DEL TEXTO: r/books
✅ Total 